This notebook covers experiments with different models and evaluation of their effectiveness.
The following approaches will be checked:
- Historical Averages
    - type: baseline
    - reason: the historical average will be evaluated to provide a minimal accuracy level expected from the model
- KNN
    - type: baseline
    - reason: based on the EDA results, no linearity between the models was revealed, so knn will be evaluated as an alternative approach to the historical averages, as it essentially finds the closest data points and computes their averages.
- Poisson Regression:
    - type: baseline
    - reason: this model type perfectly fits the regression problem because of the sold_quantity as a countable value.
- CatBoost - 2 stages: Classification and Regression
    - type: advanced
    - reason: because of the zero-inflated dataset, a classification approach to predict if there will be a sale potentially (yes/no) will help to a regression model to better understand where 0 value is expected from the beginning.

In [90]:
import pandas as pd
from sklearn.metrics import mean_absolute_error
import numpy as np
pd.set_option('display.max_rows', None)

In [91]:
df = pd.read_csv("interim_for_notebooks/complete_data_with_features.csv", parse_dates=['date'])
df.head(1)

,Unnamed: 0,flight_key,flight_number,origin,destination,date,month_name,year,weekday_name,is_weekend,...,price_bin,hist_avg_l1,hist_count_l1,hist_avg_l2,hist_count_l2,hist_avg_l3,hist_count_l3,hist_avg,hist_level_used,fold_num
0,6748,3904d3efff67caf1c3e44b230d7daca8,AB167,city_006,city_001,2026-02-15,February,2026.0,Sunday,True,...,Low,0.166667,6.0,0.166667,6.0,0.226994,163,0.166667,1.0,1


## Train / test df preparation
The last 2 weeks of DF are reserved to the test set.

In [92]:
test_cutoff = df['date'].max() - pd.Timedelta(weeks=2)
print(f"Train / test split by date: {test_cutoff}")
train_df = df[df['date'] <= test_cutoff]
test_df = df[df['date'] > test_cutoff]
print(f"Train DF size: {train_df.shape}")
# test_df = test_df.drop("sold_quantity", axis=1)
print(f"Test DF size: {test_df.shape}")

Train / test split by date: 2026-02-14 00:00:00
Train DF size: (42369, 32)
Test DF size: (5386, 32)


Model's accuracy will be evaluated  by 2 different approaches:
- technical:
    - MAE - mean absolute error
    - RMSE - root mean square error
- business - by comparing the predicted value vs the factual ones:
    - accurate_predictions: when an actual value equals to the predicted one
    - potential_waste: when an actual value is lower than the predicted one
    - potential_lost_sale: when an actual value is higher than the predicted one

In [93]:
# technical metrics evaluation
def evaluate_technical_metrics(y_true, y_predicted):

    mae = mean_absolute_error(y_true, y_predicted)
    rmse = np.sqrt((y_true - y_predicted) ** 2).mean()

    print(f"MAE:  {mae:.3f}")
    print(f"RMSE: {rmse:.3f}")

In [94]:
# business metrics evaluation
def evaluate_business_metrics(y_true, y_predicted):
    diff = y_true - y_predicted

    results = pd.DataFrame({
        "category": ["accurate_prediction", "potential_waste", "potential_lost_sale"],
        "number of records": [
            (diff == 0).sum(),
            (diff < 0).sum(),  # predicted > actual = waste
            (diff > 0).sum()   # predicted < actual = lost sale
        ],
        "share": [
            round((diff == 0).mean() * 100, 1),
            round((diff < 0).mean() * 100, 1),
            round((diff > 0).mean() * 100, 1)
        ],
        "standard deviation": [
            0,
            diff[diff < 0].mean(),  # average over-prediction
            diff[diff > 0].mean()   # average under-prediction
        ]
    })
    print(results.to_string(index=False))

## Historical averages

The historical averages are computed based on a hierarchical approach depending on the number of available records for a given level:
- Level 1: item_id - route - day_period
- Level 2: item_id - route - is_night
- Level 3: item_id - day_period

There should be at least 10 data points available in a group to compute an average for a given combination.

In [95]:
test_df = test_df.copy()
test_df['pred_baseline'] = test_df['hist_avg'].round().clip(lower=0).astype(int)

Checking the baseline model's accuracy

In [96]:
y_true = test_df["sold_quantity"]
y_predicted = test_df["pred_baseline"]
evaluate_technical_metrics(y_true, y_predicted)
evaluate_business_metrics(y_true, y_predicted)

MAE:  0.394
RMSE: 0.394
           category  number of records  share  standard deviation
accurate_prediction               3505   65.1            0.000000
    potential_waste               1034   19.2           -1.090909
potential_lost_sale                847   15.7            1.175915


## KNN - Key Nearest Neighbours

The main reason to try KNN is to narrow the specificity for the data points. Compared to the averages by hierarchical grouping, KNN allows to focus on the nearest data points (3, 5, 7) will be tried out, and then it computes the average. The main difference in the approach will be changing dynamically the threshold, so that the model can be adapted based on the business needs: to prevent wastage values or to minimize the lost sales.

The following features will be put into the model: item_id, hist_avg, route, pax_bin, day_period

The modelling process will include the following steps:
1. Categorical features preparation: ordinal, target encoding for the categorical features
2. Numerical and encoded categorical features scaling (KNN is sensitive to the scale)
3. Building the model with different hyperparameters: n_neighbors = 3, 5, 7
4. Trying out different thresholds

In [97]:
# categorical features processing
from category_encoders import TargetEncoder
from sklearn.preprocessing import OrdinalEncoder

pax_order = ['> 100', "100 - 150", "150 - 180", '180 <']
oe = OrdinalEncoder(categories=[pax_order])
train_df["pax_bin_enc"] = oe.fit_transform(train_df[["pax_bin"]])
test_df["pax_bin_enc"] = oe.transform(test_df[["pax_bin"]])

# target encoding
te = TargetEncoder(cols=["item_id", "route", "day_period"])
train_df[["item_id_enc", "route_enc", "day_period_enc"]] = te.fit_transform(train_df[["item_id", "route", "day_period"]], train_df["sold_quantity"])
test_df[["item_id_enc", "route_enc", "day_period_enc"]] = te.transform(test_df[["item_id", "route", "day_period"]], test_df["sold_quantity"])

In [98]:
# numerical features scaling
from sklearn.preprocessing import StandardScaler

features = ["hist_avg", "pax_bin_enc", "item_id_enc", "route_enc", "day_period_enc"]

scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[features])
X_test = scaler.transform(test_df[features])

y_train = train_df["sold_quantity"]
y_test = test_df["sold_quantity"]

In [99]:
# training and predicting
from sklearn.neighbors import KNeighborsRegressor

results ={}

for k in [3, 5, 7]:
    knn = KNeighborsRegressor(n_neighbors=k, metric="euclidean")
    knn.fit(X_train, y_train)
    results[k] = knn.predict(X_test)

In [100]:
# tuning threshold
import numpy as np

def apply_threshold(raw_pred, threshold=0.5):
    floored = np.floor(raw_pred)
    remainder = raw_pred - floored
    return (floored + (remainder >= threshold).astype(int)).astype(int).clip(min=0)

for k, raw_pred in results.items():
    for t in [0.3, 0.5, 0.7]:
        pred = apply_threshold(raw_pred, threshold=t)
        mae = mean_absolute_error(y_test, pred)
        print(f"\nk={k}, threshold={t:.1f} → MAE={mae:.3f}\n")
        evaluate_business_metrics(y_test, pred)


k=3, threshold=0.3 → MAE=0.616

           category  number of records  share  standard deviation
accurate_prediction               2494   46.3            0.000000
    potential_waste               2339   43.4           -1.139376
potential_lost_sale                553   10.3            1.177215

k=3, threshold=0.5 → MAE=0.435

           category  number of records  share  standard deviation
accurate_prediction               3376   62.7            0.000000
    potential_waste               1136   21.1           -1.132923
potential_lost_sale                874   16.2            1.208238

k=3, threshold=0.7 → MAE=0.372

           category  number of records  share  standard deviation
accurate_prediction               3725   69.2            0.000000
    potential_waste                561   10.4           -1.149733
potential_lost_sale               1100   20.4            1.233636

k=5, threshold=0.3 → MAE=0.511

           category  number of records  share  standard deviation
accurate_p

## CatBoost Classification + Regression


In [101]:
# feature preparation
cat_features = ["item_id", "route", "day_period", "pax_bin"]
num_features = ["hist_avg"]
features = cat_features + num_features

In [102]:
# classification + regression
train_df["target_cls"] = (train_df["sold_quantity"] > 0).astype(int)
X_train = train_df[features]
X_test = test_df[features]

y_train_cls = train_df["target_cls"]
y_train_reg = train_df[train_df["target_cls"] == 1]["sold_quantity"]
X_train_reg = train_df[train_df["target_cls"] == 1][features]

In [103]:
# training
from catboost import CatBoostClassifier, CatBoostRegressor

#classifier
cls_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    cat_features=cat_features,
    eval_metric='F1',
    class_weights=[1, 2],
    random_seed=42,
    verbose=100
)
cls_model.fit(X_train, y_train_cls)

# regressor
reg_model = CatBoostRegressor(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    cat_features=cat_features,
    eval_metric='MAE',
    random_seed=42,
    verbose=100
)
reg_model.fit(X_train_reg, y_train_reg)

0:	learn: 0.8492345	total: 11.4ms	remaining: 5.71s
100:	learn: 0.8510875	total: 1.15s	remaining: 4.54s
200:	learn: 0.8527926	total: 2.2s	remaining: 3.27s
300:	learn: 0.8561323	total: 3.33s	remaining: 2.2s
400:	learn: 0.8579174	total: 4.48s	remaining: 1.1s
499:	learn: 0.8591997	total: 5.7s	remaining: 0us
0:	learn: 0.6041081	total: 4.51ms	remaining: 2.25s
100:	learn: 0.4925571	total: 301ms	remaining: 1.19s
200:	learn: 0.4864880	total: 548ms	remaining: 816ms
300:	learn: 0.4834666	total: 829ms	remaining: 548ms
400:	learn: 0.4800270	total: 1.13s	remaining: 278ms
499:	learn: 0.4771461	total: 1.4s	remaining: 0us


CatBoostRegressor(cat_features=['item_id', 'route', 'day_period', 'pax_bin'], depth=6, eval_metric='MAE', iterations=500, learning_rate=0.05, loss_function='RMSE', random_seed=42, verbose=100)

In [104]:
# predictions with threshold
cls_proba = cls_model.predict_proba(X_test)[:, 1]
threshold = 0.5
cls_pred = (cls_proba >= threshold).astype(int)

reg_pred = reg_model.predict(X_test)
reg_pred = reg_pred.round().clip(0).astype(int)

final_pred = np.where(cls_pred == 0, 0, reg_pred)

In [105]:
# checking different thresholds
for t in [0.3, 0.4, 0.5, 0.6, 0.7]:
    cls_pred = (cls_proba >= t).astype(int)
    final_pred = np.where(cls_pred == 0, 0, reg_pred)

    mae = mean_absolute_error(y_test, final_pred)
    print(f"\nthreshold={t:.1f} → MAE={mae:.2f}\n")
    evaluate_business_metrics(y_test, final_pred)


threshold=0.3 → MAE=0.80

           category  number of records  share  standard deviation
accurate_prediction               1428   26.5            0.000000
    potential_waste               3723   69.1           -1.086758
potential_lost_sale                235    4.4            1.212766

threshold=0.4 → MAE=0.74

           category  number of records  share  standard deviation
accurate_prediction               1758   32.6            0.000000
    potential_waste               3371   62.6           -1.095817
potential_lost_sale                257    4.8            1.194553

threshold=0.5 → MAE=0.67

           category  number of records  share  standard deviation
accurate_prediction               2168   40.3            0.000000
    potential_waste               2923   54.3           -1.110503
potential_lost_sale                295    5.5            1.169492

threshold=0.6 → MAE=0.56

           category  number of records  share  standard deviation
accurate_prediction               